# 01 · Data Pipeline + HMM Regime Detection

**Purpose**: Build the full weekly feature state, fit and select the best Gaussian HMM,
infer full-sample regime probabilities, and merge FinBERT news sentiment into the RL state.

**Outputs** (written to `output/full_pipeline/`):
- `model_state_weekly_hmm_news.csv` — merged state table consumed by notebooks 02 and 03
- `hmm_regimes_full_sample.csv` — raw regime probabilities
- `hmm_bundle.pkl` — serialised HMM selection bundle

**Run once per data refresh.** Downstream notebooks load from saved files.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Locate repo root regardless of the working directory.
REPO_ROOT = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "full_pipeline").exists() and (_candidate / "scripts").exists():
        REPO_ROOT = _candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo root.")

PIPELINE_ROOT = REPO_ROOT / "full_pipeline"
for _p in (str(REPO_ROOT), str(PIPELINE_ROOT)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from _pipeline_utils import build_full_pipeline_artifacts, OUTPUT_DIR

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)
print("REPO_ROOT:", REPO_ROOT)


## 1 · Build HMM Artifacts

Runs three sequential steps:
1. Aggregate weekly FinBERT sentiment (SPY, TLT, GLD, VIX, TNX).
2. Grid-search HMM configurations and select the best by objective-aware criteria.
3. Refit on the development window, infer full-sample causal regime posteriors, merge into the state table.


In [ ]:
artifacts = build_full_pipeline_artifacts()

base_state      = artifacts["base_state"]
news_features   = artifacts["news_features"]
hmm_bundle      = artifacts["hmm_bundle"]
hmm_full        = artifacts["hmm_full"]
merged_state    = artifacts["merged_state"]
artifact_paths  = artifacts["artifact_paths"]

print("Artifacts written to:")
for name, path in artifact_paths.items():
    print(f"  {name}: {path.relative_to(REPO_ROOT)}")


## 2 · News Coverage & HMM Validation Summary


In [ ]:
coverage_df      = pd.DataFrame([artifacts["news_coverage"]])
hmm_validation_df = pd.DataFrame([artifacts["hmm_validation"]])

print("Weekly FinBERT coverage")
display(coverage_df)

print("HMM posterior validation")
display(hmm_validation_df)


## 3 · HMM Model Selection Results

The grid search evaluates multiple (K, n_pca, cov_type) combinations and ranks by:
- Validation log-likelihood per step (statistical fitness)
- Interpretability score (regime separation + label stability)

The pipeline chooses the best model satisfying hard filters; falls back to interpretability ranking.


In [ ]:
results_df = pd.DataFrame(
    [{k: v for k, v in r.items() if not k.startswith("_")} for r in hmm_bundle.results]
).sort_values(
    ["passes_hard_filters", "interpretability_score", "val_ll_per_step"],
    ascending=[False, False, False],
).reset_index(drop=True)

selected_rows = []
for label, result in [
    ("best_statistical",    hmm_bundle.best_statistical),
    ("best_interpretable",  hmm_bundle.best_interpretable),
    ("best_k3",             hmm_bundle.best_k3),
    ("chosen_for_pipeline", hmm_bundle.chosen),
]:
    if result is None:
        continue
    selected_rows.append({
        "role":                 label,
        "K":                    result["K"],
        "n_pca":                result["n_pca"],
        "cov_type":             result["cov_type"],
        "selected_seed":        result["selected_seed"],
        "val_ll_per_step":      result["val_ll_per_step"],
        "interpretability_score": result["interpretability_score"],
        "interpretability_tier":  result["interpretability_tier"],
    })

print("Selected candidates")
display(pd.DataFrame(selected_rows))

print("\nTop grid-search candidates")
display(results_df[[
    "K", "n_pca", "cov_type", "selected_seed",
    "val_ll_per_step", "passes_hard_filters",
    "interpretability_score", "interpretability_tier",
]].head(12))

print("\nSelection reason:", hmm_bundle.selection_reason)
print("Feature preset :", hmm_bundle.feature_preset)
print("Selection mode  :", hmm_bundle.selection_mode)


## 4 · Regime Labels & Posterior Probabilities


In [ ]:
filtered_cols = sorted(c for c in hmm_full.columns if c.startswith("filtered_prob_regime_"))
regime_counts = hmm_full["regime_filtered"].value_counts().sort_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(hmm_full["week_end"], hmm_full["regime_filtered"],
             drawstyle="steps-post", linewidth=1.5)
axes[0].set_title("Filtered Regime Labels — Full Sample")
axes[0].set_ylabel("Regime")

for col in filtered_cols:
    axes[1].plot(hmm_full["week_end"], hmm_full[col], label=col)
axes[1].set_title("Filtered Regime Posterior Probabilities")
axes[1].set_ylabel("Probability")
axes[1].set_xlabel("Week End")
axes[1].legend(loc="upper right", ncol=2, fontsize=9)

plt.tight_layout()
plt.show()

display(regime_counts.to_frame("n_weeks"))


## 5 · Merged State Summary

Verify that the RL state table has no missing regime or news rows before proceeding to training.


In [ ]:
print(f"Merged state: {merged_state.shape[0]} weeks × {merged_state.shape[1]} columns")
display(merged_state.head(3))

missing = pd.Series({
    "missing_regime_rows": int(merged_state["regime_filtered"].isna().sum()),
    "missing_news_rows":   int(
        merged_state.filter(like="news_finbert_compound_").isna().any(axis=1).sum()
    ),
})
display(missing.to_frame("count"))

# Sanity-check: regime posteriors sum to 1 every row.
post_cols = [c for c in merged_state.columns if c.startswith("filtered_prob_regime_")]
if post_cols:
    row_sums = merged_state[post_cols].sum(axis=1)
    print(f"Posterior sum range: [{row_sums.min():.6f}, {row_sums.max():.6f}]  (should be ~1.0)")
